In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install xgboost -q

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, classification_report

df_leaky = pd.read_csv('/content/drive/MyDrive/software-build-failure-prediction/data/processed/df_leaky.csv')
print(df_leaky.shape)
df_leaky.head()

(252151, 46)


,git_branch,git_prev_commit_resolution_status,gh_is_pr,gh_pull_req_num,gh_lang,gh_team_size,gh_num_issue_comments,gh_num_commit_comments,gh_num_pr_comments,git_diff_src_churn,...,tr_log_num_tests_skipped,tr_log_num_test_suites_run,tr_log_num_test_suites_ok,tr_log_num_test_suites_failed,tr_log_tests_failed,tr_log_testduration,tr_log_buildduration,tr_duration,tr_log_lan,target
0,master,merge_found,False,0.0,ruby,31,0.0,0,0.0,117,...,2.0,1.0,1.0,0.0,unknown,758.53,0.0,956.0,ruby,0
1,master,merge_found,False,0.0,ruby,6,0.0,1,0.0,2,...,8.0,1.0,1.0,0.0,unknown,159.32,0.0,391.0,ruby,0
2,master,build_found,False,0.0,ruby,32,0.0,0,0.0,0,...,2.0,1.0,1.0,0.0,features/filtering/if_and_unless.feature,693.85,0.0,886.0,ruby,1
3,master,build_found,False,0.0,ruby,32,0.0,0,0.0,0,...,2.0,1.0,1.0,0.0,unknown,697.15,0.0,846.0,ruby,0
4,master,build_found,False,0.0,ruby,32,0.0,0,0.0,0,...,2.0,1.0,1.0,0.0,unknown,694.37,0.0,846.0,ruby,0


In [3]:
df_leaky = df_leaky.sort_values('gh_build_started_at')

split_idx = int(len(df_leaky) * 0.8)
train_df = df_leaky.iloc[:split_idx]
test_df = df_leaky.iloc[split_idx:]

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Train failure rate:', train_df['target'].mean())
print('Test failure rate:', test_df['target'].mean())

Train shape: (201720, 46)
Test shape: (50431, 46)
Train failure rate: 0.22582788023002182
Test failure rate: 0.1734449049195931


In [4]:
X_train = train_df.drop(columns=['target', 'gh_build_started_at'])
y_train = train_df['target']
X_test = test_df.drop(columns=['target', 'gh_build_started_at'])
y_test = test_df['target']

print(X_train.shape, X_test.shape)

(201720, 44) (50431, 44)


In [5]:
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()
print('Categorical columns:', categorical_cols)

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col] = X_test[col].astype(str).map(lambda x: x if x in le.classes_ else le.classes_[0])
    X_test[col] = le.transform(X_test[col])
    encoders[col] = le

print('Encoding done.')

Categorical columns: ['git_branch', 'git_prev_commit_resolution_status', 'gh_lang', 'tr_log_status', 'tr_log_analyzer', 'tr_log_frameworks', 'tr_log_bool_tests_failed', 'tr_log_tests_failed', 'tr_log_lan']
Encoding done.


In [6]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print('scale_pos_weight:', scale_pos_weight)

scale_pos_weight: 3.428151205163103


In [7]:
from sklearn.preprocessing import StandardScaler

numeric_cols = X_train.select_dtypes(include='number').columns.tolist()

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print('Scaling done.')

Scaling done.


In [8]:
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

rf_model = RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

xgb_model = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)

print('All 3 models trained.')

All 3 models trained.


In [9]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    results = {
        'Model': name,
        'Pipeline': 'Leaky',
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba)
    }
    return results

results_list = []
results_list.append(evaluate_model('Logistic Regression', lr_model, X_test, y_test))
results_list.append(evaluate_model('Random Forest', rf_model, X_test, y_test))
results_list.append(evaluate_model('XGBoost', xgb_model, X_test, y_test))

results_df = pd.DataFrame(results_list)
print(results_df)

                 Model Pipeline  Precision    Recall        F1   AUC-ROC
0  Logistic Regression    Leaky   0.385350  0.784269  0.516780  0.769678
1        Random Forest    Leaky   0.819483  0.779010  0.798734  0.936568
2              XGBoost    Leaky   0.670291  0.852750  0.750591  0.947054


In [10]:
results_df.to_csv('/content/drive/MyDrive/software-build-failure-prediction/results/metrics_comparison.csv', index=False)
print('Saved leaky baseline results.')

Saved leaky baseline results.
